# Proof of concept for a RAG System on arxiv abstracts

# Architecture:
- Embeddings: sentence-transformer
- Vectors: FAISS
- PDF parsing with PyMuPDF
- Models from Huggingface

## Imports and Configs

In [1]:
# !pip install kagglehub
# !pip install pymupdf
# !pip install faiss-cpu

In [2]:
import pandas as pd
import pymupdf as fitz
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm
import faiss

import os, json, re, textwrap, time, requests
from pathlib import Path
from typing import List, Dict, Any, Tuple, Optional


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
path = Path("../Data/datasets/arxiv_data.csv")

CFG = {
    "data_path" : path,
    
    # Change these according to final chosen dataset
    "col_title" : "titles",
    "col_summaries" : "summaries",
    "col_terms" : "terms",
    
    "embed_model" : "sentence-transformers/all-MiniLM-L6-v2", # Larger alternative: "sentence-transformers/all-mpnet-base-v2"
    "embed_batch" : 256,
    
    "llm_model" : "Qwen/Qwen2.5-3B-Instruct", # Larger alternative: "Qwen/Qwen2.5-7B-Instruct", "meta-llama/Llama-3.1-8B-Instruct", "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B"
    "llm_max_tokens" : 512,
    "llm_temperature" : 0.7,
    "llm_load_in_4bit" : False,
    
    "top_k" : 5,
    
    # -- Caches -- (to avoid recomputation during development)
    "index_path" : "arxiv.faiss", # FAISS index file
    "meta_path" : "arxiv_meta.json", # Metadata for each paper (title, summary, terms, embedding vector)
}

# DEVICE = "cuda" if torch.cuda.is_available() else "cpu" # Non M-series Mac
DEVICE = "mps" if torch.backends.mps.is_available() else DEVICE # M-series Mac
print(f"Using device: {DEVICE}")

Using device: mps


## Preprocessing of Dataset

In [4]:
def load_dataset(cfg):
    path = cfg["data_path"]
    df = pd.read_csv(path)
    
    df.columns = [c.strip().lower() for c in df.columns]
    required = [cfg["col_title"], cfg["col_summaries"]] # Change this according to final chosen dataset
    
    df = df.dropna(subset=required).reset_index(drop=True)
    df["all_text"] = (
        df[cfg["col_title"]].str.strip() + " " + df[cfg["col_summaries"]].str.strip()
    )
    
    print(f"Loaded {len(df)} papers.")
    return df


df = load_dataset(CFG)
df.head(2)

Loaded 51774 papers.


,titles,summaries,terms,all_text
0,Survey on Semantic Stereo Matching / Semantic ...,Stereo matching is one of the widely used tech...,"['cs.CV', 'cs.LG']",Survey on Semantic Stereo Matching / Semantic ...
1,FUTURE-AI: Guiding Principles and Consensus Re...,The recent advancements in artificial intellig...,"['cs.CV', 'cs.AI', 'cs.LG']",FUTURE-AI: Guiding Principles and Consensus Re...


## Embeddings

In [5]:
def load_embed_model(cfg):
    model = SentenceTransformer(cfg["embed_model"], device=DEVICE)
    print(f"  Embedding dim: {model.get_sentence_embedding_dimension()}")
    return model

embed_model = load_embed_model(CFG)

  Embedding dim: 384


## FAISS Index

Used for similarity search and clustering of vectors

In [6]:
def embed_texts(texts, model, batch_size):
    all_embeddings = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Embedding batches"):
        batch = texts[i : i + batch_size]
        embs = model.encode(batch, convert_to_numpy=True, show_progress_bar=False)
        embs = embs / (np.linalg.norm(embs, axis=1, keepdims=True) + 1e-10) # Normalize 
        all_embeddings.append(embs.astype(np.float32))
        
    return np.vstack(all_embeddings)

In [7]:
def build_faiss_index(embeddings):
    dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)  # Inner Product for cosine similarity
    index.add(embeddings)
    print(f"FAISS index built with {index.ntotal} vectors.")
    
    return index


def save_index(index, meta, cfg):
    faiss.write_index(index, cfg["index_path"])
    with open(cfg["meta_path"], "w") as f:
        json.dump(meta, f)
    print(f"Index and metadata saved to {cfg['index_path']} and {cfg['meta_path']}.")
    

def load_index(cfg):
    index = faiss.read_index(cfg["index_path"])
    with open(cfg["meta_path"], "r") as f:
        meta = json.load(f)
    print(f"Index and metadata loaded from {cfg['index_path']} and {cfg['meta_path']}.")
    return index, meta

In [8]:
def get_or_build_index(df, embed_model, cfg):
    if Path(cfg["index_path"]).exists() and Path(cfg["meta_path"]).exists():
        print("Found existing index and metadata. Loading...")
        return load_index(cfg)
    
    print("No existing index found. Building new index...")
    texts = df["all_text"].tolist()
    embeddings = embed_texts(texts, embed_model, cfg["embed_batch"])

    index = build_faiss_index(embeddings)
    
    meta = [
        {
            "idx" : int(i),
            "title" : str(row[cfg["col_title"]]),
            "summary" : str(row[cfg["col_summaries"]]),
            "terms" : str(row[cfg["col_terms"]]) if cfg["col_terms"] in row else "",
        }
        for i, row in df.iterrows()
    ]
    
    save_index(index, meta, cfg)
    return index, meta

index, meta = get_or_build_index(df, embed_model, CFG)

No existing index found. Building new index...


Embedding batches: 100%|██████████| 203/203 [01:34<00:00,  2.15it/s]


FAISS index built with 51774 vectors.
Index and metadata saved to arxiv.faiss and arxiv_meta.json.


## Retrieval

In [9]:
def retrieve(query, embed_model, index, meta, top_k):
    """
    Embed a query string and return the top-k most similar papers.
    Each result dict contains: title, abstract, terms, score.
    """
    
    q_emb = embed_model.encode([query], convert_to_numpy=True)
    q_emb = q_emb / (np.linalg.norm(q_emb, axis=1, keepdims=True) + 1e-10) # Normalize
    
    scores, ids = index.search(q_emb, top_k)

    results = []
    for score, idx in zip(scores[0], ids[0]):
        if idx == -1:
            continue
        entry = meta[idx].copy()
        entry["score"] = float(score)
        results.append(entry)
    return results

# Quick sanity check
sample = retrieve("transformer attention mechanism NLP", embed_model, index, meta, top_k=3)
for r in sample:
    print(f"[{r['score']:.3f}] {r['title']}")

[0.702] Adaptive Attention Span in Transformers
[0.697] ETC: Encoding Long and Structured Inputs in Transformers
[0.697] ETC: Encoding Long and Structured Inputs in Transformers


## LLM for answer generation

In [10]:
def load_llm(cfg):
    model_id = cfg["llm_model"]
    print(f"Loading LLM: {model_id} (This might take a while...)")
    
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    load_kwargs = {"torch_dtype": torch.float16 if DEVICE == "mps" else torch.float32} # M-series Mac
    # load_kwargs = {"torch_dtype": torch.float16 if DEVICE == "cuda" else torch.float32} # Non M-series Mac
    
    if DEVICE == "mps": # or cuda
        load_kwargs["device_map"] = "auto"
    if cfg.get("llm_load_in_4bit") and DEVICE == "mps": # or cuda
        from transformers import BitsAndBytesConfig
        load_kwargs["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True)
        load_kwargs.pop("torch_dtype", None)

    model = AutoModelForCausalLM.from_pretrained(model_id, **load_kwargs)
    if DEVICE == "cpu":
        model = model.to(DEVICE)

    model.eval()
    print("LLM loaded.")
    return tokenizer, model

llm_tokenizer, llm_model = load_llm(CFG)

Loading LLM: Qwen/Qwen2.5-3B-Instruct (This might take a while...)


`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 2/2 [00:01<00:00,  1.08it/s]


LLM loaded.
